# Lab: Phân loại chữ số viết tay MNIST với mạng ANN

Ở bài trước ta đã xây một ANN nhỏ phân loại điểm trong/ngoài hình tròn. Bây giờ ta nâng cấp lên một bài thật sự: nhận diện chữ số 0-9 từ ảnh 28×28 pixel.

**Mục tiêu của bài lab này:**
- Hiểu cách "đẩy" một ảnh vào mạng ANN: ảnh 28×28 → vector 784 chiều.
- Nắm được pipeline chuẩn của một bài phân loại trong PyTorch: `Dataset → DataLoader → Model → Loss → Optimizer → Train loop → Eval`.
- Tránh được một cái bẫy *cực kỳ phổ biến*: dùng `nn.Softmax` chung với `nn.CrossEntropyLoss` (sẽ giải thích kỹ ở phần model).
- Vẽ được learning curve và confusion matrix để biết model sai ở đâu.

Cuối bài có 5 bài tập để các em tự luyện.

## 1. Bộ dữ liệu MNIST

MNIST là tập dữ liệu "hello world" của Deep Learning: 70.000 ảnh chữ số viết tay (60k train + 10k test), mỗi ảnh là một ma trận 28×28 pixel grayscale (giá trị 0-255). Nhiệm vụ: nhìn ảnh, đoán xem là số mấy (0-9).

Vì đây là bài 10 lớp (multi-class), ta sẽ:
- Dùng **CrossEntropyLoss** (chuẩn cho multi-class).
- Đầu ra của model là **10 logits** (một số thực cho mỗi lớp), KHÔNG cần softmax cuối.
- Đo độ chính xác bằng cách chọn lớp có logit cao nhất.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np

# Cố định seed để chạy đi chạy lại ra cùng một kết quả.
# Khi báo cáo bài lab, các em luôn nên đặt seed.
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Đang dùng device:', device)

## 2. Tải dữ liệu và chuẩn hoá

Pixel gốc có giá trị 0-255. Khi đưa vào mạng, ta nên đưa về khoảng nhỏ hơn để gradient ổn định. Có 2 mức chuẩn hoá thường dùng:

1. **`ToTensor()`**: chia cho 255, đưa về [0, 1].
2. **`Normalize(mean, std)`**: trừ mean, chia std → mean=0, std=1.

Với MNIST, mean ≈ 0.1307 và std ≈ 0.3081 (đã tính sẵn từ tập train). Dùng cả hai bước này giúp model hội tụ nhanh hơn rõ rệt.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = MNIST(root='./data', train=True,  transform=transform, download=True)
test_dataset  = MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f'Số ảnh train: {len(train_dataset)}')
print(f'Số ảnh test : {len(test_dataset)}')
print(f'Kích thước 1 ảnh: {train_dataset[0][0].shape}  (channel x H x W)')

### Xem thử vài ảnh

Trước khi train bao giờ cũng phải nhìn dữ liệu. Nhỡ ảnh bị xoay ngược, hay nhãn bị lệch thì train mãi cũng không ra.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    # img có shape (1, 28, 28); .squeeze() bỏ chiều channel để vẽ.
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'Nhãn: {label}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Xây dựng model — và một cái bẫy phải tránh

Kiến trúc đơn giản: `Flatten → Linear(784→128) → ReLU → Linear(128→10)`. Đầu ra 10 số.

### Lưu ý quan trọng — bẫy: KHÔNG đặt Softmax ở cuối khi dùng CrossEntropyLoss

Đây là lỗi mà 9/10 bạn lần đầu học PyTorch hay mắc. Vì sao?

Công thức `nn.CrossEntropyLoss` thực chất là:
$$
\text{Loss} = -\log\!\Big(\frac{e^{z_y}}{\sum_j e^{z_j}}\Big) = -\log(\text{softmax}(z)_y)
$$

Tức là `CrossEntropyLoss` đã **tự tính softmax bên trong** rồi. Nếu mình đặt thêm `nn.Softmax(dim=1)` ở cuối model thì hoá ra ta đang tính `softmax(softmax(z))` — hai lần liên tiếp. Hậu quả:
- Loss không bao giờ về gần 0 (kẹt quanh 1.4-1.5 thay vì xuống 0.05).
- Gradient bị bóp nhỏ → train chậm.
- Accuracy thấp hơn khoảng 1-2% so với khi làm đúng.

**Quy tắc nhớ đời:** Forward của model trả về *logits* (số thực, chưa qua softmax). Việc biến logits thành xác suất là việc của loss function (lúc train) hoặc của lúc inference (dùng `torch.softmax` thủ công).

In [ ]:
class ANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)
        # KHÔNG có Softmax ở đây — CrossEntropyLoss sẽ lo phần đó.

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)   # trả về logits, shape (batch, 10)
        return x

model = ANN().to(device)
print(model)

# Đếm số tham số để biết model nặng cỡ nào.
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Tổng số tham số: {n_params:,}')

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## 4. Vòng lặp huấn luyện

Một vài chi tiết quan trọng:

- **`model.train()` đầu mỗi epoch**: bật chế độ train (Dropout, BatchNorm sẽ hoạt động đúng). Model này không có hai thứ đó nhưng tạo thói quen tốt.
- **`model.eval()` + `torch.no_grad()` khi đánh giá**: tắt việc lưu graph để tính gradient, đỡ tốn bộ nhớ và nhanh hơn.
- **Lưu loss trung bình của cả epoch**, không phải loss của batch cuối. Loss của batch cuối nhảy lung tung không phản ánh xu hướng.
- **Đo train accuracy ngay trong lúc train** (cộng dồn `correct` qua các batch), không cần chạy lại toàn bộ tập train sau mỗi epoch.

In [ ]:
def evaluate(model, loader):
    # Trả về (loss trung bình, accuracy) của model trên một loader bất kỳ.
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

num_epochs = 10
train_loss_history, train_acc_history = [], []
test_loss_history,  test_acc_history  = [], []

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Cộng dồn: nhân loss với batch_size để tính trung bình theo mẫu, không theo batch.
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total
    test_loss, test_acc = evaluate(model, test_loader)

    train_loss_history.append(train_loss)
    train_acc_history.append(train_acc)
    test_loss_history.append(test_loss)
    test_acc_history.append(test_acc)

    print(f'Epoch {epoch+1:2d}/{num_epochs}  '
          f'train_loss={train_loss:.4f}  train_acc={train_acc*100:.2f}%   '
          f'test_loss={test_loss:.4f}  test_acc={test_acc*100:.2f}%')

## 5. Vẽ learning curve

Vẽ cả train và test trên cùng một đồ thị. Nếu thấy train acc tăng mà test acc bắt đầu giảm — đó là dấu hiệu **overfitting**, lúc đó cần thêm regularization (Dropout, weight decay) hoặc dừng sớm.

In [ ]:
epochs = range(1, num_epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_loss_history, 'o-', label='Train')
axes[0].plot(epochs, test_loss_history,  's-', label='Test')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss qua các epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, [a*100 for a in train_acc_history], 'o-', label='Train')
axes[1].plot(epochs, [a*100 for a in test_acc_history],  's-', label='Test')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)'); axes[1].set_title('Accuracy qua các epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Confusion matrix — model sai ở đâu?

Accuracy tổng cho ta một con số. Nhưng nó không nói cho ta biết: model nhầm số 4 thành số 9 nhiều, hay nhầm số 7 thành số 1. Đây là lúc confusion matrix lên ngôi.

Hàng = nhãn thật, cột = nhãn dự đoán. Đường chéo càng đậm thì model càng tốt; ô ngoài đường chéo cho biết model hay nhầm cặp số nào.

In [ ]:
model.eval()
cm = torch.zeros(10, 10, dtype=torch.long)
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu()
        for t, p in zip(labels, preds):
            cm[t.item(), p.item()] += 1

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm.numpy(), cmap='Blues')
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel('Dự đoán'); ax.set_ylabel('Nhãn thật'); ax.set_title('Confusion matrix trên tập test')
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j].item(), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=8)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Xem top-3 cặp nhầm nhiều nhất
off_diag = cm.clone()
off_diag.fill_diagonal_(0)
flat_idx = torch.topk(off_diag.flatten(), 3).indices
print('Top-3 cặp model hay nhầm nhất (thật → đoán: số lần):')
for idx in flat_idx:
    i, j = idx.item() // 10, idx.item() % 10
    print(f'  {i} → {j}: {cm[i, j].item()} lần')

## 7. Xem model đoán thử vài ảnh

Lấy ngẫu nhiên 10 ảnh test, in ảnh kèm nhãn thật và nhãn đoán. Ảnh nào đoán sai sẽ tô đỏ tiêu đề.

In [ ]:
model.eval()
indices = np.random.choice(len(test_dataset), 10, replace=False)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
with torch.no_grad():
    for ax, idx in zip(axes.flat, indices):
        img, label = test_dataset[idx]
        logits = model(img.unsqueeze(0).to(device))
        pred = logits.argmax(dim=1).item()
        # un-normalize lại để hiển thị cho dễ nhìn
        ax.imshow(img.squeeze() * 0.3081 + 0.1307, cmap='gray')
        color = 'green' if pred == label else 'red'
        ax.set_title(f'Thật: {label} | Đoán: {pred}', color=color)
        ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Tóm lại

Ta vừa đi qua một pipeline phân loại ảnh hoàn chỉnh:
1. Tải MNIST + chuẩn hoá pixel.
2. Định nghĩa ANN trả về **logits** (không softmax cuối).
3. Train với Adam + CrossEntropyLoss, theo dõi train/test loss & acc.
4. Đánh giá bằng confusion matrix để biết model sai ở cặp nào.

Model ANN này đạt khoảng **97-98% test accuracy**. Ở bài CNN tiếp theo, ta sẽ thấy CNN có thể đẩy con số này lên >99% với ít tham số hơn. Vì sao? Vì CNN tận dụng được tính chất *không gian* của ảnh, còn ANN thì "flatten" ảnh thành vector, vứt mất thông tin lân cận giữa các pixel.

## 9. Bài tập về nhà

Mỗi bài đều có gợi ý — không có lời giải, các em tự code rồi báo cáo kết quả.

### Bài 1: Sâu hơn có tốt hơn không?
Thêm 1 lớp ẩn nữa (`784 → 256 → 128 → 10`). So sánh test accuracy với model 1 lớp ẩn ban đầu.
- *Gợi ý:* chỉ cần thêm `self.fc_mid = nn.Linear(256, 128)` và sửa `fc1` thành `Linear(784, 256)`.
- Báo cáo: số tham số tăng bao nhiêu, accuracy có tăng tương xứng không?

### Bài 2: Ảnh hưởng của learning rate
Train lại model gốc với 3 giá trị `lr ∈ {1e-2, 1e-3, 1e-4}`, vẽ 3 đường loss trên cùng đồ thị.
- *Gợi ý:* viết một hàm `train(lr)` trả về `loss_history`, gọi 3 lần.
- Quan sát: lr nào hội tụ nhanh, lr nào dao động, lr nào quá chậm?

### Bài 3: Thêm Dropout
Chèn `nn.Dropout(p=0.3)` giữa hai lớp Linear. Train 15 epoch và so sánh khoảng cách `train_acc - test_acc` trước và sau khi thêm Dropout.
- *Gợi ý:* Dropout chỉ hoạt động khi `model.train()`; ở `model.eval()` nó tự tắt.
- Mục tiêu: chứng minh Dropout giúp giảm overfitting (gap nhỏ lại).

### Bài 4: Phân tích sai số theo lớp
Từ confusion matrix, tính accuracy *theo từng lớp* (per-class accuracy). Lớp nào model làm tệ nhất?
- *Gợi ý:* `cm[i, i] / cm[i].sum()` cho lớp i.
- Bonus: in ra 5 ảnh trong lớp tệ nhất bị đoán sai, xem có giống chữ nào khác không.

### Bài 5: Vẽ trọng số của fc1
Lấy `model.fc1.weight` (shape `[128, 784]`), reshape mỗi hàng thành ảnh 28×28 và vẽ 16 hàng đầu tiên thành lưới 4×4.
- *Gợi ý:* `weights = model.fc1.weight.detach().cpu().view(128, 28, 28)`.
- Quan sát: các "feature" mà neuron đầu tiên học được trông như thế nào?